In [ ]:
from pathlib import Path
import pandas as pd
from glob import glob
from tqdm import tqdm
import os

In [ ]:
src_path = '../../data/stock_data/raw/'
dst_path = '../../data/stock_data/processed/'
ai_ready_path = '../../data/stock_data/ai_ready/'




In [ ]:
# 모델 입력에 필요한 원본 컬럼과 파생 계산용 컬럼
source_columns = [
    'symbol',
    'timestamp_kst',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'trade_strength',
    'aggressive_buy_volume',
    'aggressive_sell_volume',
    'session_vwap',
    'return_pct',
    'bid_ask_imbalance',
    'avg_bid_size',
    'avg_ask_size',
    'avg_spread_pct',
    'ema9',
    'ema20',
    'trade_notional_from_ticks',
]

output_columns = [
    'symbol',
    'timestamp_kst',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'trade_strength',
    'aggressive_buy_volume',
    'aggressive_sell_volume',
    'session_vwap',
    'return_pct',
    'bid_ask_imbalance',
    'avg_bid_size',
    'avg_ask_size',
    'avg_spread_pct',
    'ema9',
    'ema20',
    'volume_ratio_ma20',
    'vwap_gap_ratio',
    'ema_momentum',
    'aggressive_buy_ratio',
    'capital_inflow_ratio_ma20',
    'spread_ratio_ma20',
    'orderbook_imbalance_strength',
]


def safe_divide(numerator, denominator):
    # 0으로 나눈 값은 inf 대신 NaN으로 남긴다.
    return numerator.div(denominator.where(denominator.ne(0)))
src_files_list = glob(src_path + '**/*_enriched.csv')

if not src_files_list:
    raise FileNotFoundError(f'enriched CSV를 찾지 못했습니다: {src_path}')

dst_root = Path(dst_path)
dst_root.mkdir(parents=True, exist_ok=True)

# processed 폴더는 평면 구조로 유지한다. 원본 파일의 날짜와 run ID를 파일명에 보존한다.
source_paths = [Path(file_path) for file_path in sorted(src_files_list)]
expected_output_names = {
    source_file.name.removesuffix('_enriched.csv') + '_features.csv'
    for source_file in source_paths
}

# 이전 실행에서 만든 종목 통합 파일 또는 사라진 원본의 stale feature 파일을 제거한다.
for existing_path in dst_root.glob('*_features.csv'):
    if existing_path.name not in expected_output_names:
        existing_path.unlink()

save_records = []
processed_files_list = []
warmup_bars = 20
total_input_rows = 0
total_saved_rows = 0

for source_file in tqdm(source_paths, desc='Build per-file features'):
    frame = pd.read_csv(source_file, usecols=source_columns, low_memory=False)
    frame['timestamp_kst'] = pd.to_datetime(frame['timestamp_kst'], errors='raise')
    frame = frame.sort_values('timestamp_kst').reset_index(drop=True)

    if frame['symbol'].nunique() != 1:
        raise ValueError(f'한 파일에 여러 종목이 있습니다: {source_file}')
    if frame.duplicated(['symbol', 'timestamp_kst']).any():
        raise ValueError(f'동일 symbol/timestamp_kst 중복이 있습니다: {source_file}')

    # rolling은 다른 날짜 파일과 연결하지 않고 현재 종목·거래일 파일 내부에서만 계산한다.
    volume_mean20 = frame['volume'].rolling(20, min_periods=20).mean()
    notional_mean20 = frame['trade_notional_from_ticks'].rolling(20, min_periods=20).mean()
    spread_mean20 = frame['avg_spread_pct'].rolling(20, min_periods=5).mean()

    frame['volume_ratio_ma20'] = safe_divide(frame['volume'], volume_mean20)
    frame['vwap_gap_ratio'] = safe_divide(
        frame['close'] - frame['session_vwap'],
        frame['session_vwap'],
    )
    frame['ema_momentum'] = frame['ema9'] - frame['ema20']
    frame['aggressive_buy_ratio'] = safe_divide(
        frame['aggressive_buy_volume'],
        frame['volume'],
    )
    frame['capital_inflow_ratio_ma20'] = safe_divide(
        frame['trade_notional_from_ticks'],
        notional_mean20,
    )
    frame['spread_ratio_ma20'] = safe_divide(
        frame['avg_spread_pct'],
        spread_mean20,
    )
    frame['orderbook_imbalance_strength'] = (
        frame['bid_ask_imbalance'] - 0.5
    ).abs()

    # rolling 지표 준비 구간인 파일별 최초 20개 봉은 최종 데이터에서 제거한다.
    result = frame.iloc[warmup_bars:][output_columns].reset_index(drop=True).copy()

    # 정상 범위가 0 이상인 결측 컬럼은 학습용 sentinel -1로 저장한다.
    missing_value_columns = [
        'trade_strength',
        'avg_spread_pct',
        'avg_bid_size',
        'avg_ask_size',
        'bid_ask_imbalance',
        'spread_ratio_ma20',
        'orderbook_imbalance_strength',
    ]
    result[missing_value_columns] = result[missing_value_columns].fillna(-1.0)
    output_name = source_file.name.removesuffix('_enriched.csv') + '_features.csv'
    output_path = dst_root / output_name
    result.to_csv(output_path, index=False)

    symbol = str(result['symbol'].iloc[0])
    total_input_rows += len(frame)
    total_saved_rows += len(result)
    processed_files_list.append(str(output_path))
    save_records.append({
        'symbol': symbol,
        'rows': len(result),
        'start': result['timestamp_kst'].min(),
        'end': result['timestamp_kst'].max(),
        'source_file': source_file.name,
        'output_file': output_path.name,
    })

save_summary = pd.DataFrame(save_records)
print(f'입력 파일: {len(source_paths):,}개')
print(f'원본 전체 행: {total_input_rows:,}개')
print(f'제거한 초기 봉: {total_input_rows - total_saved_rows:,}개')
print(f'저장 전체 행: {total_saved_rows:,}개')
print(f'고유 종목: {save_summary["symbol"].nunique():,}개')
print(f'저장 파일: {len(save_summary):,}개')
print(f'저장 위치: {dst_root.resolve()}')
display(save_summary.head(20))


In [ ]:
import numpy as np
import shutil

# 30개 봉을 입력으로 사용하고, 마지막 봉(t)의 종가에서 진입했다고 가정한다.
input_length = 30
volatility_window = 30       # t 시점까지 최근 30개 로그수익률
prediction_horizon = 10      # t 이후 최대 10개 봉 동안 관찰
take_profit_sigma = 2.0
stop_loss_sigma = 1.0
buy_fee_rate = 0.001         # 매수 수수료 및 거래비용 0.1%
sell_fee_rate = 0.001        # 매도 수수료 및 거래비용 0.1%
other_round_trip_cost_rate = 0.001  # 기타 왕복 비용/슬리피지 가정 0.1%
target_net_return = 0.010    # 모든 비용 차감 후 목표 순수익 1.0%
tick_size = 0.01             # 대상 종목의 실제 호가 단위에 맞게 조정
min_take_profit_ticks = 3
min_stop_loss_ticks = 2
min_window_notional_usd = 100_000  # 입력 30봉의 총 거래대금 하한

ai_ready_root = Path(ai_ready_path)
input_root = ai_ready_root / 'input'
ai_ready_root.mkdir(parents=True, exist_ok=True)

# 재실행 시 이전 input 폴더와 label.csv를 제거한 뒤 깨끗하게 다시 만든다.
if input_root.exists():
    shutil.rmtree(input_root)
input_root.mkdir(parents=True, exist_ok=False)
(ai_ready_root / 'label.csv').unlink(missing_ok=True)

# 앞 셀을 실행하지 않고 이 셀만 다시 실행해도 processed 파일을 찾는다.
feature_files = (
    [Path(path) for path in processed_files_list]
    if 'processed_files_list' in globals()
    else sorted(Path(dst_path).glob('*_features.csv'))
)
if not feature_files:
    raise FileNotFoundError(f'feature CSV를 찾지 못했습니다: {dst_path}')

label_records = []
for feature_file in tqdm(feature_files, desc='Build AI-ready windows'):
    frame = pd.read_csv(feature_file, low_memory=False)
    frame['timestamp_kst'] = pd.to_datetime(frame['timestamp_kst'], errors='raise')
    frame = frame.sort_values('timestamp_kst').reset_index(drop=True)

    # 미래 정보 없이 t 시점까지의 종가만으로 변동성을 계산한다.
    log_return = np.log(frame['close'] / frame['close'].shift(1))
    sigma = log_return.rolling(volatility_window, min_periods=volatility_window).std(ddof=1)
    last_entry_index = len(frame) - prediction_horizon - 1

    for t in range(input_length - 1, last_entry_index + 1):
        input_window = frame.iloc[t - input_length + 1:t + 1]
        if len(input_window) != input_length:
            continue

        # 입력 30봉과 미래 라벨 10봉에 누락된 1분봉이 있으면 샘플에서 제외한다.
        input_time_diff = input_window['timestamp_kst'].diff().dropna()
        future_window = frame.iloc[t:t + prediction_horizon + 1]
        future_time_diff = future_window['timestamp_kst'].diff().dropna()
        one_minute = pd.Timedelta(minutes=1)
        if not input_time_diff.eq(one_minute).all():
            continue
        if not future_time_diff.eq(one_minute).all():
            continue

        # processed 데이터에는 실제 tick 거래대금이 없으므로 close × volume proxy를 사용한다.
        window_notional_usd = (input_window['close'] * input_window['volume']).sum()
        if not np.isfinite(window_notional_usd) or window_notional_usd < min_window_notional_usd:
            continue

        sigma_t = sigma.iloc[t]
        if not np.isfinite(sigma_t) or sigma_t <= 0:
            continue

        entry_price = float(frame.at[t, 'close'])
        if not np.isfinite(entry_price) or entry_price <= 0:
            continue

        # avg_spread_pct는 0.3 == 0.3% 형식이다. 결측 sentinel(-1)은 비용에 넣지 않는다.
        spread_pct = frame.at[t, 'avg_spread_pct']
        spread_rate = (
            float(spread_pct) / 100
            if np.isfinite(spread_pct) and spread_pct >= 0
            else 0.0
        )
        round_trip_cost_rate = (
            buy_fee_rate
            + sell_fee_rate
            + other_round_trip_cost_rate
            + spread_rate
        )
        tick_rate = tick_size / entry_price

        # sigma는 로그수익률이므로 먼저 단순 수익률로 바꾼 뒤 비용/틱 기준과 비교한다.
        volatility_take_profit_rate = np.expm1(take_profit_sigma * sigma_t)
        volatility_stop_loss_rate = 1 - np.exp(-stop_loss_sigma * sigma_t)
        take_profit_rate = max(
            volatility_take_profit_rate,
            round_trip_cost_rate + target_net_return,
            min_take_profit_ticks * tick_rate,
        )
        stop_loss_rate = max(
            volatility_stop_loss_rate,
            spread_rate,
            min_stop_loss_ticks * tick_rate,
        )
        if stop_loss_rate >= 1:
            continue

        # TP는 위쪽 틱, SL은 아래쪽 틱으로 맞춰 barrier를 보수적으로 넓힌다.
        raw_take_profit_price = entry_price * (1 + take_profit_rate)
        raw_stop_loss_price = entry_price * (1 - stop_loss_rate)
        take_profit_price = np.ceil(raw_take_profit_price / tick_size) * tick_size
        stop_loss_price = np.floor(raw_stop_loss_price / tick_size) * tick_size
        gross_take_profit_rate = take_profit_price / entry_price - 1
        gross_stop_loss_rate = 1 - stop_loss_price / entry_price
        expected_net_take_profit_rate = gross_take_profit_rate - round_trip_cost_rate
        expected_net_stop_loss_rate = -(gross_stop_loss_rate + round_trip_cost_rate)
        label, exit_reason, exit_offset = 0, 'timeout', prediction_horizon

        for offset in range(1, prediction_horizon + 1):
            future_row = frame.iloc[t + offset]
            take_profit_hit = future_row['high'] >= take_profit_price
            stop_loss_hit = future_row['low'] <= stop_loss_price

            # 같은 1분봉에서 둘 다 닿으면 봉 내부 순서를 모르므로 손절을 우선한다.
            if stop_loss_hit:
                label, exit_reason, exit_offset = -1, 'stop_loss', offset
                break
            if take_profit_hit:
                label, exit_reason, exit_offset = 1, 'take_profit', offset
                break

        sample_id = f'{feature_file.stem}__t{t:06d}'
        input_file = input_root / f'{sample_id}.csv'
        input_window[output_columns].to_csv(input_file, index=False)
        label_records.append({
            'sample_id': sample_id,
            'input_file': str(Path('input') / input_file.name),
            'symbol': str(frame.at[t, 'symbol']),
            'entry_timestamp_kst': frame.at[t, 'timestamp_kst'],
            'entry_price': entry_price,
            'window_notional_usd': window_notional_usd,
            'sigma': sigma_t,
            'spread_rate': spread_rate,
            'tick_rate': tick_rate,
            'round_trip_cost_rate': round_trip_cost_rate,
            'take_profit_price': take_profit_price,
            'stop_loss_price': stop_loss_price,
            'gross_take_profit_rate': gross_take_profit_rate,
            'gross_stop_loss_rate': gross_stop_loss_rate,
            'expected_net_take_profit_rate': expected_net_take_profit_rate,
            'expected_net_stop_loss_rate': expected_net_stop_loss_rate,
            'label': label,
            'exit_reason': exit_reason,
            'exit_offset': exit_offset,
            'source_file': feature_file.name,
        })

labels = pd.DataFrame(label_records)
label_path = ai_ready_root / 'label.csv'
labels.to_csv(label_path, index=False)

print(f'입력 샘플: {len(labels):,}개')
print(f'샘플당 입력 봉: {input_length}개')
print(f'입력 저장 위치: {input_root.resolve()}')
print(f'라벨 저장 위치: {label_path.resolve()}')
display(labels['label'].value_counts(dropna=False).sort_index().rename('count'))
display(labels.head())
